In [1]:
from sqlalchemy import create_engine, MetaData, Table, select, insert
from sqlalchemy.exc import SQLAlchemyError
from dotenv import load_dotenv
load_dotenv()
import os
from tqdm import tqdm
import pandas as pd
import regex as re


color.csv = https://www.kaggle.com/datasets/ehsanzafari/colors-csv?resource=download <br>
 - added:
    - green
    - orange

In [2]:
with open('colors.csv') as f:
    df_colors = pd.read_csv(f,header=None)

In [3]:
def create_db_connection():
    DB_HOST = os.getenv("DB_HOST")
    DB_NAME = os.getenv("DB_NAME")
    DB_USER = os.getenv("DB_USER")
    DB_PASSWORD = os.getenv("DB_PASSWORD")
    engine = create_engine('postgresql+pg8000://'+DB_USER+':'+DB_PASSWORD+'@'+DB_HOST+':5432/'+DB_NAME)
    return engine


def get_tags():
    try:
        engine = create_db_connection()
        df_systag = pd.read_sql_table('oc_systemtag', engine)


    except SQLAlchemyError as e:
        print(f"An error occurred: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    return df_systag

In [4]:
df_systag = get_tags()
df_systag.drop('etag', axis=1, inplace=True)
df_systag.drop('color', axis=1, inplace=True)
df_systag.drop('editable', axis=1, inplace=True)
df_systag.drop('visibility', axis=1, inplace=True)

is color?

In [5]:
df_systag['is_color'] = df_systag['name'].isin(df_colors[0].tolist())

In [6]:
df_systag

,id,name,is_color
0,819,flower,False
1,820,nature,False
2,821,still,False
3,822,life,False
4,823,vase,False
...,...,...,...
761,1580,gentle,False
762,1581,tender,False
763,1582,emotive,False
764,1583,room,False


adjectives?

In [7]:
from nltk.corpus import wordnet as nwn

In [8]:
#all_nouns = [word for synset in nwn.all_synsets('n') for word in synset.lemma_names()]
all_adjects = [word for synset in nwn.all_synsets(nwn.ADJ) for word in synset.lemma_names()]


In [9]:
for index, row in df_systag.iterrows():
    if row['name'] not in df_colors[0].tolist():
        df_systag.loc[index,['adjective']] = True if row['name'] in all_adjects else False
    else:
        df_systag.loc[index,['adjective']] = False
df_systag

,id,name,is_color,adjective
0,819,flower,False,False
1,820,nature,False,False
2,821,still,False,True
3,822,life,False,False
4,823,vase,False,False
...,...,...,...,...
761,1580,gentle,False,True
762,1581,tender,False,True
763,1582,emotive,False,True
764,1583,room,False,False


hyponyms?

In [10]:
import wn, wn.taxonomy
ewn = wn.Wordnet('ewn:2020')
dog = ewn.synsets('dog', pos='n')[0]

In [11]:
for index, row in df_systag[(df_systag['is_color'] == False) & (df_systag['adjective'] == False)].head(2).iterrows():
    print(row['name'])
    item = ewn.synsets(row['name'], pos='n')[0]
    for hyponyms in wn.taxonomy.hypernym_paths(item):
        for i, hyponym in enumerate(hyponyms):
            #print(' '* i, hyponym, hyponym.lemmas()[0])
            print(' '* i, hyponym.lemmas()[0])



flower
 angiosperm
  spermatophyte
   vascular plant
    plant
     being
      animate thing
       whole
        object
         physical entity
          entity
nature
 quality
  attribute
   abstraction
    entity


In [12]:
for index, row in df_systag[(df_systag['is_color'] == False)].iterrows():
    #print(row['name'])
    try:
        item = ewn.synsets(row['name'], pos='n')[0]
        for hyponyms in wn.taxonomy.hypernym_paths(item):
            #print(len(hyponyms))
            lvl3 = len(hyponyms) -3
            if lvl3 <=0:
                lvl3 = 0
            #print('Level 3 hypernym:', hyponyms[lvl3].lemmas()[0])
            df_systag.loc[index,['lvl3_hyponym']] = hyponyms[lvl3].lemmas()[0]
    except:
        df_systag.loc[index,['lvl3_hyponym']] = df_systag.loc[index,['name']]

In [13]:
df_systag

,id,name,is_color,adjective,lvl3_hyponym
0,819,flower,False,False,object
1,820,nature,False,False,attribute
2,821,still,False,True,object
3,822,life,False,False,attribute
4,823,vase,False,False,object
...,...,...,...,...,...
761,1580,gentle,False,True,NaN
762,1581,tender,False,True,amount
763,1582,emotive,False,True,NaN
764,1583,room,False,False,object


In [14]:
for index, row in df_systag[(df_systag['is_color'] == False)][['id','name']].iterrows():
    try:
        item = ewn.synsets(row['name'], pos='n')[0]
        for hyponyms in wn.taxonomy.hypernym_paths(item):
            hype_list = []
            for i, hyponym in enumerate(hyponyms):
                hype_list.append(hyponym.lemmas()[0])
            #print(hype_list)
            df_systag.loc[index,['hyponym_all']] = str(hype_list)
    except:
        item = row['name']
#df_systag.drop('name', axis=1, inplace=True)

In [15]:
df_systag

,id,name,is_color,adjective,lvl3_hyponym,hyponym_all
0,819,flower,False,False,object,"['angiosperm', 'spermatophyte', 'vascular plan..."
1,820,nature,False,False,attribute,"['quality', 'attribute', 'abstraction', 'entity']"
2,821,still,False,True,object,"['exposure', 'representation', 'creation', 'ar..."
3,822,life,False,False,attribute,"['existence', 'state', 'attribute', 'abstracti..."
4,823,vase,False,False,object,"['jar', 'vessel', 'container', 'instrumentalit..."
...,...,...,...,...,...,...
761,1580,gentle,False,True,NaN,NaN
762,1581,tender,False,True,amount,"['monetary system', 'standard', 'metric', 'amo..."
763,1582,emotive,False,True,NaN,NaN
764,1583,room,False,False,object,"['area', 'construction', 'artifact', 'whole', ..."


In [16]:
df_hyperesearch = df_systag[['id','is_color','adjective','lvl3_hyponym','hyponym_all']]
df_hyperesearch

,id,is_color,adjective,lvl3_hyponym,hyponym_all
0,819,False,False,object,"['angiosperm', 'spermatophyte', 'vascular plan..."
1,820,False,False,attribute,"['quality', 'attribute', 'abstraction', 'entity']"
2,821,False,True,object,"['exposure', 'representation', 'creation', 'ar..."
3,822,False,False,attribute,"['existence', 'state', 'attribute', 'abstracti..."
4,823,False,False,object,"['jar', 'vessel', 'container', 'instrumentalit..."
...,...,...,...,...,...
761,1580,False,True,NaN,NaN
762,1581,False,True,amount,"['monetary system', 'standard', 'metric', 'amo..."
763,1582,False,True,NaN,NaN
764,1583,False,False,object,"['area', 'construction', 'artifact', 'whole', ..."


In [23]:
engine = create_db_connection() 
df_hyperesearch['id'] = df_hyperesearch['id'].astype(int)
df_hyperesearch[['is_color','adjective']] = df_hyperesearch[['is_color','adjective']].astype(bool)
bre_advance_search =  pd.read_sql_table('bre_advance_search', engine)



/var/folders/7j/rm0vdpf11g14s7ckw_8gxmx80000gn/T/ipykernel_70440/3935041885.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hyperesearch['id'] = df_hyperesearch['id'].astype(int)
/var/folders/7j/rm0vdpf11g14s7ckw_8gxmx80000gn/T/ipykernel_70440/3935041885.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hyperesearch[['is_color','adjective']] = df_hyperesearch[['is_color','adjective']].astype(bool)


In [25]:
bre_advance_search.concat(df_hyperesearch).drop_duplicates(subset=['id'], keep='last')

AttributeError: 'DataFrame' object has no attribute 'concat'

In [22]:
df_hyperesearch.to_sql('bre_advance_search', engine, if_exists='append')

InterfaceError: (pg8000.exceptions.InterfaceError) in failed transaction block
(Background on this error at: https://sqlalche.me/e/20/rvf5)